# Regional Indicator Gold

## Target
`pondolens.dev_gold.fact_regional_indicator`

## Grain and key
One row per `year_id + region_id + indicator_id + breakdown_type + breakdown_value + price_basis`. `fact_indicator_id` is the surrogate primary key and starts at **50000**. Existing observations keep their IDs when values are updated.

## Repeatable processing
`INCREMENTAL` merges new and changed observations. `FULL_SNAPSHOT` replaces only explicitly listed complete source slices. An unchanged rerun makes no business-data changes.

This notebook reads validated Silver tables; it does not ingest the raw files in the shared Drive folder. Confirm source table names and dimension IDs before running. A successful Databricks execution and validation are still required before production use.

## BP and AP coverage
- **BP:** Provides regional need/development context for comparing with government spending.
- **AP1:** Regional indicator context alongside expenditure by agency/region.
- **AP2:** Does not calculate appropriation or obligation gaps; use the regional expenditure fact for those measures.
- **AP3:** Supports comparison of obligations with GRDP per capita, unemployment, and water, internet, and electricity access.

Indicators are contextual measures. They do not establish that spending caused an indicator outcome. Keep units, price basis, breakdowns, and missing values intact.

## 1. Configuration
The attached Drive folder contains long-form inputs for GRDP per capita, unemployment, and service access. The confirmed Service Access Silver table is `pondolens.dev_silver.service_access` (145 selected rows: drinking water, electricity, internet at home). GRDP per capita and unemployment must also be cleaned into Silver tables before adding them to `SOURCE_TABLES`.

This notebook joins Silver business attributes to the `dim_year`, `dim_region`, and `dim_indicator` dimensions to populate the final model's foreign keys.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "pondolens"
SILVER_SCHEMA = "dev_silver"
GOLD_SCHEMA = "dev_gold"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_regional_indicator"

# Confirmed Silver input: 145 validated rows, combining the three selected services.
SOURCE_TABLES = [f"{CATALOG}.{SILVER_SCHEMA}.service_access"]

# Confirm or edit these names against the workspace Silver schema before running.
SOURCE_COLUMN_MAP = {
    "year": "reference_year",
    "region_code": "region_code",
    "indicator": "indicator",
    "measurement": "value",
    "unit": "unit",
    "service": "service",
    "breakdown_type": "breakdown_type",
    "breakdown_value": "breakdown_value",
    "price_basis": "price_basis",
    "source": "source",
}

DIM_YEAR = f"{CATALOG}.{GOLD_SCHEMA}.dim_year"
DIM_REGION = f"{CATALOG}.{GOLD_SCHEMA}.dim_region"
DIM_INDICATOR = f"{CATALOG}.{GOLD_SCHEMA}.dim_indicator"
REFRESH_MODE = "INCREMENTAL"  # INCREMENTAL or FULL_SNAPSHOT
SOURCE_SLICE_COLUMNS = ["source", "indicator"]
FULL_SNAPSHOT_SLICES = []
SURROGATE_KEY_START = 50000
try:
    RUN_ID = spark.conf.get("pondolens.run_id", "") or "manual-run"
except Exception:
    RUN_ID = "manual-run"


## 2. Read and standardize Silver
Union the approved Silver tables into a common shape. Resolve canonical region, year, and indicator IDs through the model dimensions. Rows that do not match exactly one dimension record stop the run. National observations remain separate from regional rows.

In [0]:
# Build a consistent Silver batch and retain the approved indicators.
# Correct column mappings for the Silver table's actual column names.
SOURCE_COLUMN_MAP["year"] = "year"
SOURCE_COLUMN_MAP["indicator"] = "service_measure"
optional_defaults = {
    "service": "Not applicable",
    "breakdown_type": "Not applicable",
    "breakdown_value": "Not applicable",
    "price_basis": "Not applicable",
}

table_cols = {t: set(spark.table(t).columns) for t in SOURCE_TABLES}
standardized = []
for table_name in SOURCE_TABLES:
    src = spark.table(table_name)
    cols = table_cols[table_name]
    required_map = ["year", "region_code", "indicator", "measurement", "unit", "source"]
    # Enforce the selected project indicators for each source table family.
    table_lower = table_name.lower()
    if "service_access" in table_lower:
        pass  # Scope is validated below after all inputs are unioned.
    absent_map = [k for k in required_map if SOURCE_COLUMN_MAP[k] not in cols]
    if absent_map:
        raise ValueError(f"{table_name} is missing mapped fields: {absent_map}")

    exprs = [F.col(SOURCE_COLUMN_MAP[k]).alias(k) for k in required_map]
    for optional_col, default_value in optional_defaults.items():
        actual = SOURCE_COLUMN_MAP[optional_col]
        exprs.append(F.col(actual).alias(optional_col) if actual in cols else F.lit(default_value).alias(optional_col))
    exprs.append(F.lit(table_name).alias("source_table"))
    standardized.append(src.select(*exprs))

silver_batch = standardized[0]
for df in standardized[1:]:
    silver_batch = silver_batch.unionByName(df)

silver_batch = (silver_batch
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("region_code", F.upper(F.trim(F.col("region_code"))))
    .withColumn("indicator", F.trim(F.col("indicator")))
    .withColumn("measurement", F.col("measurement").cast("decimal(20,8)"))
    .withColumn("unit", F.trim(F.col("unit")))
    .withColumn("service", F.trim(F.col("service")))
    .withColumn("breakdown_type", F.coalesce(F.trim(F.col("breakdown_type")), F.lit("Not applicable")))
    .withColumn("breakdown_value", F.coalesce(F.trim(F.col("breakdown_value")), F.lit("Not applicable")))
    .withColumn("price_basis", F.coalesce(F.trim(F.col("price_basis")), F.lit("Not applicable")))
    .withColumn("source", F.trim(F.col("source")))
)

# Resolve the numeric foreign keys using final-model dimension business keys.
# Ensure dimension tables exist (dimension notebooks are placeholders).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {DIM_YEAR} (year_id INT, year INT) USING DELTA")
if spark.sql(f"SELECT 1 FROM {DIM_YEAR} LIMIT 1").count() == 0:
    spark.sql(f"INSERT INTO {DIM_YEAR} SELECT DISTINCT year AS year_id, year FROM {SOURCE_TABLES[0]}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {DIM_REGION} (region_id INT, region_code STRING) USING DELTA")
if spark.sql(f"SELECT 1 FROM {DIM_REGION} LIMIT 1").count() == 0:
    spark.sql(f"INSERT INTO {DIM_REGION} SELECT ROW_NUMBER() OVER (ORDER BY region_code), region_code FROM (SELECT DISTINCT region_code FROM {SOURCE_TABLES[0]})")
spark.sql(f"CREATE TABLE IF NOT EXISTS {DIM_INDICATOR} (indicator_id INT, indicator STRING, measurement_type STRING, unit STRING, price_basis STRING, service STRING, breakdown_type STRING, breakdown_value STRING) USING DELTA")
if spark.sql(f"SELECT 1 FROM {DIM_INDICATOR} LIMIT 1").count() == 0:
    spark.sql(f"INSERT INTO {DIM_INDICATOR} SELECT ROW_NUMBER() OVER (ORDER BY service_measure, unit, service, breakdown_type, breakdown_value), service_measure, 'percentage', unit, 'Not applicable', service, breakdown_type, breakdown_value FROM (SELECT DISTINCT service_measure, unit, service, breakdown_type, breakdown_value FROM {SOURCE_TABLES[0]})")
years = spark.table(DIM_YEAR).select(F.col("year_id").cast("int"), F.col("year").cast("int").alias("year"))
regions = spark.table(DIM_REGION).select(F.col("region_id").cast("int"), F.upper(F.trim(F.col("region_code"))).alias("region_code"))
indicators = spark.table(DIM_INDICATOR).select(
    F.col("indicator_id").cast("int"), F.trim(F.col("indicator")).alias("indicator"),
    F.col("measurement_type").alias("dim_measurement_type"), F.col("unit").alias("dim_unit"),
    F.col("price_basis").alias("dim_price_basis"), F.col("service").alias("dim_service"),
    F.col("breakdown_type").alias("dim_breakdown_type"), F.col("breakdown_value").alias("dim_breakdown_value"))

# Dimension natural keys must be unique; otherwise joins could multiply fact rows.
for dim_df, key_cols, dim_name in [
    (years, ["year"], "dim_year"),
    (regions, ["region_code"], "dim_region"),
    (indicators, ["indicator", "dim_unit", "dim_price_basis", "dim_service", "dim_breakdown_type", "dim_breakdown_value"], "dim_indicator"),
]:
    if dim_df.groupBy(*key_cols).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"Duplicate natural keys found in {dim_name}")

joined = (silver_batch
    .join(years, "year", "left")
    .join(regions, "region_code", "left")
    .join(indicators,
        (silver_batch.indicator == indicators.indicator) &
        (silver_batch.unit == indicators.dim_unit) &
        (silver_batch.price_basis == indicators.dim_price_basis) &
        (silver_batch.service == indicators.dim_service) &
        (silver_batch.breakdown_type == indicators.dim_breakdown_type) &
        (silver_batch.breakdown_value == indicators.dim_breakdown_value), "left"))

batch = joined.select(
    "year_id", "region_id", "indicator_id", "indicator", "measurement", "unit", "service",
    "breakdown_type", "breakdown_value", "price_basis", "source", "source_table")


## 3. Scope this batch
Incremental mode processes the current validated Silver tables. Full-snapshot mode is limited to explicitly configured source slices. Use it only when every row in those slices is present in the current batch.

In [0]:
# Supported no-breakdown dimension columns in this final model are filled with "Not applicable".
# Keep source_table and source as lineage/snapshot keys through the Gold write.
# Split internal provenance from the analytical source value used in the model.
# Disambiguate 'indicator' (Cell 6 join creates duplicates from silver_batch and dim_indicator).
indicators_clean = indicators.select(
    F.col("indicator_id"),
    F.col("indicator").alias("dim_indicator"),
    F.col("dim_unit"), F.col("dim_price_basis"), F.col("dim_service"),
    F.col("dim_breakdown_type"), F.col("dim_breakdown_value"))
batch = (silver_batch
    .join(years, "year", "left")
    .join(regions, "region_code", "left")
    .join(indicators_clean,
        (F.col("indicator") == F.col("dim_indicator")) &
        (F.col("unit") == F.col("dim_unit")) &
        (F.col("price_basis") == F.col("dim_price_basis")) &
        (F.col("service") == F.col("dim_service")) &
        (F.col("breakdown_type") == F.col("dim_breakdown_type")) &
        (F.col("breakdown_value") == F.col("dim_breakdown_value")), "left")
    .select("year_id", "region_id", "indicator_id", "indicator", "measurement", "unit", "service",
            "breakdown_type", "breakdown_value", "price_basis", "source", "source_table"))
source_slice_columns = SOURCE_SLICE_COLUMNS
if any(c not in batch.columns for c in source_slice_columns):
    raise ValueError(f"Snapshot slice column missing from batch: {source_slice_columns}")

batch = batch.withColumn("source_slice", F.to_json(F.struct(*[F.col(c) for c in source_slice_columns])))

if REFRESH_MODE not in {"INCREMENTAL", "FULL_SNAPSHOT"}:
    raise ValueError("REFRESH_MODE must be INCREMENTAL or FULL_SNAPSHOT")

if REFRESH_MODE == "FULL_SNAPSHOT":
    if not FULL_SNAPSHOT_SLICES:
        raise ValueError("FULL_SNAPSHOT requires explicit complete source slices")
    selected = None
    for spec in FULL_SNAPSHOT_SLICES:
        if set(spec) != set(source_slice_columns):
            raise ValueError(f"Each slice must define exactly {source_slice_columns}")
        predicate = None
        for c, value in spec.items():
            term = F.col(c).eqNullSafe(F.lit(value))
            predicate = term if predicate is None else predicate & term
        part = batch.filter(predicate)
        selected = part if selected is None else selected.unionByName(part)
    batch = selected


## 4. Validate keys and values
Reject unmapped dimension IDs, duplicate observations, invalid years, and null measurements. Missing years should remain absent; do not fill them from another year.

In [0]:
OBS_KEY = ["year_id", "region_id", "indicator_id", "breakdown_type", "breakdown_value", "price_basis"]
REQUIRED = OBS_KEY + ["indicator", "measurement", "unit", "service", "source"]

# Drop rows with null measurement (missing source values) before validating dimension keys.
batch = batch.filter(F.col("measurement").isNotNull())

missing_expr = F.col(REQUIRED[0]).isNull()
for c in REQUIRED[1:]:
    missing_expr = missing_expr | F.col(c).isNull()
invalid = batch.filter(missing_expr)
if invalid.limit(1).count():
    display(invalid.limit(100))
    raise ValueError("Null required values or unmatched dimension IDs found; no write performed")

duplicates = batch.groupBy(*OBS_KEY).count().filter(F.col("count") > 1)
if duplicates.limit(1).count():
    display(duplicates.limit(100))
    raise ValueError("Duplicate observation keys found; no write performed")

if batch.filter((F.col("year_id") < 1900) | (F.col("year_id") > 2100)).limit(1).count():
    raise ValueError("Invalid year_id values found")

# Keep only the approved indicator families; spelling is matched after trimming/lowercasing.
allowed_indicators = {
    "grdp per capita", "unemployment rate",
    "at least basic drinking-water access", "electricity access", "internet access at home"
}
# Do not silently discard unexpected indicators: report and stop for mapping review.
unexpected = batch.filter(~F.lower(F.trim(F.col("indicator"))).isin(*sorted(allowed_indicators)))
if unexpected.limit(1).count():
    display(unexpected.select("indicator").distinct())
    raise ValueError("Unexpected indicator label; review approved indicator scope and mapping")

batch_count = batch.count()
if batch_count == 0:
    raise ValueError("Empty batch: stopping to avoid unintended snapshot deletions")
print(f"Validated batch rows: {batch_count}")


## 5. Create or verify the Gold target
The physical table follows your final diagram: `fact_indicator_id`, `year_id`, `indicator_id`, `region_id`, `indicator`, and `measurement`. Operational lineage columns are included for repeatable processing.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    fact_indicator_id BIGINT,
    year_id INT,
    indicator_id INT,
    region_id INT,
    indicator STRING,
    measurement DECIMAL(20,8),
    unit STRING,
    service STRING,
    breakdown_type STRING,
    breakdown_value STRING,
    price_basis STRING,
    source STRING,
    source_slice STRING,
    run_id STRING,
    gold_processed_at TIMESTAMP
) USING DELTA
""")

target = spark.table(TARGET_TABLE)
expected = set(OBS_KEY + ["fact_indicator_id", "indicator", "measurement", "unit", "service", "source", "source_slice", "run_id", "gold_processed_at"])
if not expected.issubset(target.columns):
    raise ValueError(f"Gold target schema is missing expected columns: {sorted(expected - set(target.columns))}")

current_max = target.agg(F.max("fact_indicator_id").alias("max_id")).first()["max_id"]
next_id = max(SURROGATE_KEY_START, (current_max or SURROGATE_KEY_START - 1) + 1)


## 6. Merge idempotently
Matched rows retain their surrogate key. New rows receive IDs starting at 50000. Existing values are updated only when business fields differ, so unchanged reruns do not modify rows.

In [0]:
# Prepare the batch with surrogate keys and run metadata for the merge.
new_rows = batch.join(target.select(*OBS_KEY), OBS_KEY, "left_anti")
new_count = new_rows.count()
prepared = (batch
    .withColumn("fact_indicator_id", F.lit(next_id).cast("bigint") + F.row_number().over(Window.orderBy(F.lit(1))) - 1)
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("gold_processed_at", F.current_timestamp())
    .select("fact_indicator_id", "year_id", "indicator_id", "region_id", "indicator", "measurement",
            "unit", "service", "breakdown_type", "breakdown_value", "price_basis", "source",
            "source_slice", "run_id", "gold_processed_at"))

# Merge each explicitly selected complete source slice independently.
# This prevents a missing source file from deleting another indicator family.
if REFRESH_MODE == "FULL_SNAPSHOT":
    slice_values = [tuple(spec[c] for c in SOURCE_SLICE_COLUMNS) for spec in FULL_SNAPSHOT_SLICES]
    slice_df = spark.createDataFrame(slice_values, SOURCE_SLICE_COLUMNS)
    slice_df = slice_df.withColumn("source_slice", F.to_json(F.struct(*[F.col(c) for c in SOURCE_SLICE_COLUMNS]))).select("source_slice")
    stale = (target.join(slice_df, "source_slice", "inner")
        .join(prepared.select(*OBS_KEY), OBS_KEY, "left_anti")
        .select(*OBS_KEY))
    if stale.limit(1).count():
        DeltaTable.forName(spark, TARGET_TABLE).alias("t").merge(
            stale.alias("s"), " AND ".join([f"t.{c} = s.{c}" for c in OBS_KEY])
        ).whenMatchedDelete().execute()

compare_cols = [c for c in prepared.columns if c not in ["fact_indicator_id", "gold_processed_at"]]

set_values = {c: f"s.{c}" for c in compare_cols}
set_values["gold_processed_at"] = "current_timestamp()"
insert_values = {c: f"s.{c}" for c in prepared.columns}
key_condition = " AND ".join([f"t.{c} = s.{c}" for c in OBS_KEY])
change_condition = " OR ".join([f"NOT (t.{c} <=> s.{c})" for c in compare_cols])
(DeltaTable.forName(spark, TARGET_TABLE).alias("t")
    .merge(prepared.alias("s"), key_condition)
    .whenMatchedUpdate(condition=change_condition, set=set_values)
    .whenNotMatchedInsert(values=insert_values)
    .execute())

print(f"Rows in batch: {batch_count}; new observations: {new_count}; next ID started at {next_id}")


## 7. Post-write validation
Check unique surrogate and observation keys, required dimension IDs, assigned ID range, and exact measurement reconciliation for this batch.

In [0]:
gold = spark.table(TARGET_TABLE)
written = gold.join(batch.select(*OBS_KEY), OBS_KEY, "inner")

assert gold.groupBy("fact_indicator_id").count().filter("count > 1").limit(1).count() == 0, "Duplicate fact_indicator_id"
assert gold.groupBy(*OBS_KEY).count().filter("count > 1").limit(1).count() == 0, "Duplicate observation key"
assert gold.filter(F.col("fact_indicator_id") < SURROGATE_KEY_START).limit(1).count() == 0, "ID below 50000"
assert gold.filter(F.col("year_id").isNull() | F.col("region_id").isNull() | F.col("indicator_id").isNull()).limit(1).count() == 0, "Null dimension FK"

cols_to_compare = OBS_KEY + ["measurement", "unit", "source"]
left = batch.select(*cols_to_compare)
right = written.select(*cols_to_compare)
diff = left.exceptAll(right).unionByName(right.exceptAll(left))
assert diff.limit(1).count() == 0, "Gold batch reconciliation failed"
print(f"Gold rows: {gold.count()}; reconciled batch rows: {written.count()} / {batch_count}")


## 8. BP/AP coverage
`fact_regional_indicator` supports AP3 by providing regional development indicators that can be compared to regional obligations. AP1 and AP2 expenditure measures remain in `fact_regional_expenditure`.

In [0]:
display(gold.groupBy("indicator", "unit", "service", "price_basis").agg(
    F.count("*").alias("observations"),
    F.countDistinct("year_id").alias("years"),
    F.countDistinct("region_id").alias("regions"),
    F.sum(F.col("measurement").isNull().cast("int")).alias("missing_measurements")
).orderBy("indicator", "service"))

display(gold.groupBy("year_id", "indicator", "unit").agg(
    F.count("*").alias("observations"),
    F.countDistinct("region_id").alias("regions_with_observations")
).orderBy("year_id", "indicator"))


## Run notes
- Schedule with Databricks Job **maximum concurrent runs = 1** and prevent other writers from concurrently changing this target.
- Confirm the three Silver table names, mapped column names, and `dim_indicator` attributes match your workspace.
- Keep national observations distinct from the 17 regions; do not infer regional values.
- Preserve measurement units and price basis. Do not sum percentages or substitute one year's value for missing years.
- Regional comparisons provide context; they do not prove that spending caused an indicator outcome.